# Clase 11: Aprendizaje No Supervisado 1 - Clustering

**MDS7202: Laboratorio de Programación Científica para Ciencia de Datos**


## Objetivos de la Clase


- Comprender cuál es la utilidad de las técnicas de clustering y cuándo aplicarlas.
- Analizar y comparar diversos tipos de algoritmos de clustering (K-Means, K-Modes, K-Prototypes, DBSCAN, Jerárquico, GMM).
- Aprender a evaluar la calidad de un clustering en ausencia de etiquetas (Silhouette, método del codo, BIC/AIC).
- Comprender las limitaciones y supuestos de cada algoritmo y saber elegir entre ellos según el tipo de datos.
- Interpretar y caracterizar los clusters obtenidos a partir de perfiles estadísticos y composición por categorías.

## Machine Learning 🤔



<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/14-Feature-Engineering-Parte-I/machine_learning.png?raw=true' width=700 />


## Clustering

Clustering es la tarea que consiste en agrupar observaciones similares en grupos llamados *clusters*. La idea es que los grupos solo contengan información similar.
Es una tarea usual al realizar Análisis Exploratorio de Datos (EDA), ya que permite encontrar de forma automatizada grupos de observaciones similares.

Ya que no es necesario que el dataset esté etiquetado, es una técnica de aprendizaje no-supervisado.

<div align='center'>
    <img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/17-Clustering/clustering.png?raw=true' width=800/>
</div>

<div align='center'>
    <span>Ejemplo de Clustering. Fuente: <a href='https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html'>Comparación de Clustering en Scikit-Learn.</a></span>
</div>

### ¿Para qué sirve en la práctica?

El clustering aparece en una gran variedad de dominios reales:

- 🛒 **Segmentación de clientes**: Agrupar usuarios según sus hábitos de consumo para personalizar recomendaciones.
- 🧬 **Bioinformática**: se agrupan genes con patrones de expresión similares para descubrir funciones biológicas.
- 🌐 **Detección de comunidades**: identificación de grupos en redes sociales o redes de citas académicas.
- 🔒 **Ciberseguridad**: detección de comportamientos anómalos en tráfico de red al separar el tráfico "normal" del sospechoso.
- 🗺️ **Geografía y urbanismo**: agrupación de zonas de una ciudad según indicadores socioeconómicos o de movilidad.
- 🎵 **Caracterización musical** ← *nuestro problema de hoy*: identificar géneros o estilos a partir de descriptores de audio.

## Tipos de Clustering

Existen varias técnicas de clustering, las cuales se pueden clasificar en las siguientes categorías:


|  | **Particional** | **Jerárquico** | **Densidad** | **Difuso** |
|---|---|---|---|---|
| Descripción | Divide los datos en clusters sin traslape, tal que cada dato está en un solo grupo y en ningún otro. | Agrupa ejemplos al ir estableciendo jerarquías entre estos, de tal manera que los datos son organizados como un árbol. | Agrupa puntos en regiones densas del espacio de características, separadas por regiones de baja densidad. Permite detectar clusters de forma arbitraria y outliers. | Cada objeto pertenece a cada cluster con un peso de pertenencia entre 0 y 1. |
| Ejemplos | K-Means | Aglomerativo, Divisivo | DBSCAN | Mixtura de Gaussianas |

## Clustering en `Scikit-learn`

Scikit-learn ofrece una gran gama de algoritmos de clustering para explorar.





<div align='center'>
    <img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/17-Clustering/cluster_comparison.png?raw=true' width=800/>
</div>

<div align='center'>
    <span>Ejemplo de los distintos métodos de Clustering ofrecidos por Scikit-Learn. </span>
    <br>
    Fuente: <a href='https://scikit-learn.org/stable/modules/clustering.html'>Clustering en Scikit-Learn.</a>
</div>

---

## Problema de Hoy: 🎸🤘 Caracterización Musical 🎼🎵 

<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/17-Clustering/spotify.png?raw=true' width=200/>
</div>

 
    
Los atributos son: 

- `key`: escala de la canción. 0 = C, 1 = C♯/D♭, 2 = D...  [Mas información](https://en.wikipedia.org/wiki/Pitch_class).
- `modo`: 1 si la escala es mayor, 0 si es menor.
- `time_signature`: cuántos pulsos hay en cada compás. (4, 3,...).
- `loudness`: Volumen de la canción (rango -60, 0).


- `acousticness`: Probabilidad de que la canción sea solo acústica.
- `danceability`: Describe que tan bailable es la canción. (rango 0, 1).
- `energy`: Mide que tan energética es una canción (rango 0, 1).
- `instrumentalness`: Probabilidad de que la canción **no** contenga voces (valores altos → canción instrumental, sin letra).
- `liveness`: Probabilidad de que la canción fuese grabada en vivo.
- `speechiness`: Probabilidad de que la canción sea exclusivamente vocal (ejemplo: podcast : 1). 
- `valence`: Sentimiento de la canción (rango 0, 1). 1 -> felicidad, alegría, euforia. 0 -> Tristeza, enojo, depresión.
- `tempo` : Pulsos por minuto de la canción (BPM). 
- `genre`: Género de la canción.



### Análisis Exploratorio de Datos

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv(
    "https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/descriptores_musica.csv"
)
df.head(5)

In [ ]:
print("Géneros de música del Dataset:", df["genre"].unique().tolist())
print("Artistas del Dataset:", df["artist"].unique().tolist())

In [ ]:
df.query("artist == 'The Beatles'")

In [ ]:
df.describe().round(2)

In [ ]:
def get_sample(df: pd.DataFrame, idx: int):
    """
    Obtiene un ejemplo y lo formatea como columna.
    """
    ejemplo = (
        df.loc[
            idx,
            [
                "danceability",
                "energy",
                "speechiness",
                "acousticness",
                "instrumentalness",
                "valence",
                "name",
                "artist",
                "genre",
            ],
        ]
        .to_frame()
        .reset_index()
    )
    ejemplo.columns = ["Descriptor", "Valor"]
    return ejemplo

In [ ]:
# pueden cambiar el índice de alguno de estos ejemplos para
# mostrar otra canción en la visualización
ejemplos = [
    get_sample(df, 483),
    get_sample(df, 105),
    get_sample(df, 575),
    get_sample(df, 810),
]

#### Spider/Radar Chart

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "polar"}, {"type": "polar"}],
        [{"type": "polar"}, {"type": "polar"}],
    ],
    subplot_titles=[
        f"{ejemplo.loc[7, 'Valor']} - {ejemplo.loc[6, 'Valor']} ({ejemplo.loc[8, 'Valor']})"
        for ejemplo in ejemplos
    ],
)

for i, ejemplo in enumerate(ejemplos):
    fig.add_trace(
        go.Scatterpolar(
            r=ejemplo.loc[0:5, "Valor"],
            theta=ejemplo.loc[0:5, "Descriptor"],
            fill="toself",
        ),
        col=i // 2 + 1,
        row=i % 2 + 1,
    )

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=False,
    title="ScatterPolar/Radar/Spider Chart/ Descripción de Ejemplos",
    height=800,
)

fig.show()

#### Histogramas

In [ ]:
px.histogram(df, x="duration_ms", marginal="box")

In [ ]:
px.histogram(df, x="loudness", marginal="box")

In [ ]:
px.histogram(df, x="tempo", marginal="box")

In [ ]:
dt_to_hists = df.loc[
    :,
    [
        "danceability",
        "energy",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "valence",
        "liveness",
        "genre",
    ],
].melt(id_vars=["genre"], var_name="variable", value_name="valor")

px.histogram(
    dt_to_hists, x="valor", color="variable", facet_col="variable", facet_col_wrap=4
).update_layout(showlegend=False)

### Correlaciones

In [ ]:
corr = df.loc[
    :,
    [
        "danceability",
        "energy",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "valence",
    ],
].corr(numeric_only=True)
px.imshow(corr)

---

## Preparando los Datos

Para esta clase usaremos los siguientes atributos:

In [ ]:
df_ = df.loc[
    :,
    [
        "danceability",
        "energy",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "valence",
        "liveness",
        "duration_ms",
        "loudness",
        "tempo",
        "name",
        "genre",
    ],
]

df_

In [ ]:
df_.describe()

### `MinMaxScaler`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

ct = ColumnTransformer(
    transformers=[("MinMax", MinMaxScaler(), ["duration_ms", "loudness", "tempo"])],
    remainder="passthrough",
)


pipe = Pipeline(steps=[("Preprocesamiento", ct)])

In [ ]:
features_to_scale = df_.iloc[:, :-2]  # eliminar nombre y género

In [ ]:
# transformamos el resultado de la transformación a un dataframe:
scaled_features = pd.DataFrame(
    pipe.fit_transform(features_to_scale), columns=features_to_scale.columns
)

scaled_features

In [ ]:
scaled_features.describe()

---

## `UMAP` - Proyectamos con UMAP

**UMAP** (*Uniform Manifold Approximation and Projection*) es un algoritmo de reducción de dimensionalidad no lineal. A diferencia de PCA, UMAP preserva tanto la estructura local (puntos cercanos siguen cercanos) como parte de la estructura global del espacio de alta dimensionalidad.

Lo usamos aquí para **visualizar en 2D** los datos (que tienen 10 atributos), lo que nos permite ver intuitivamente si los algoritmos de clustering están encontrando grupos coherentes. 

> ⚠️ Importante: el scatter 2D es solo una proyección para visualización. El clustering se calcula sobre los **datos originales en alta dimensión**, no sobre la proyección.

In [ ]:
from umap import UMAP

proyector = UMAP(random_state=88, n_neighbors=20, min_dist=0.15, n_components=2)
projections = proyector.fit_transform(scaled_features)

In [ ]:
projections.shape

In [ ]:
# este dataframe lo estaremos usando para graficar de aquí en adelante
fig_df = pd.concat(
    [
        df.loc[:, ["name", "artist", "genre"]],
        scaled_features,
        pd.DataFrame(projections, columns=["x", "y"]),
    ],
    axis=1,
)

fig_df

In [ ]:
def get_scatter(fig_df, color_col):
    fig = px.scatter(
        fig_df,
        x="x",
        y="y",
        color=color_col,
        hover_name=df["artist"] + " - " + df["name"],
        # labels={"genre": "Género Musical"},
        hover_data=[
            "danceability",
            "energy",
            "speechiness",
            "acousticness",
            "instrumentalness",
            "valence",
            "duration_ms",
            "loudness",
            "tempo",
            "genre",
        ],
        range_x=(fig_df["x"].min() - 1, fig_df["x"].max() + 1),
        range_y=(fig_df["y"].min() - 1, fig_df["y"].max() + 1),
    )
    return fig


get_scatter(fig_df, "genre")

---

## Algoritmos de Clustering

> **Pregunta motivadora** 🤔: Mirando el scatter plot anterior coloreado por género, ¿cómo agruparían estas canciones si tuvieran que hacerlo **manualmente**, sin conocer los géneros? ¿Qué criterio usarían?

## K-Means

K-Means es un algoritmo de clustering **particional**: divide los datos en $K$ grupos sin traslape, buscando que cada punto quede lo más cerca posible al centroide de su cluster.

La idea central es encontrar $K$ centros (centroides) que minimicen la **suma de errores cuadrados (SSE)** — es decir, la suma de las distancias al cuadrado entre cada punto y el centroide de su cluster:

$$SSE = \sum_{k=1}^{K} \sum_{x \in C_k} d(c_k,\, x)^2$$

donde $c_k$ es el centroide del cluster $C_k$ y $d$ es la distancia euclideana $d(x,y) = \sqrt{\sum_{i}(x_i - y_i)^2}$.

### Algoritmo

---

    1. Seleccionar K centroides iniciales (al azar o con K-Means++).
    2. Repetir:
        a. Asignar cada punto al centroide más cercano.
        b. Recalcular cada centroide como la media de los puntos asignados.
    3. Hasta que los centroides no cambien (o el cambio sea menor a un umbral).

---

**¿Por qué converge?** En cada iteración, tanto el paso de asignación como el de actualización reducen o mantienen la SSE. Como la SSE es acotada inferiormente (≥ 0), el algoritmo siempre converge. Sin embargo, puede hacerlo en un **mínimo local**, por lo que se recomienda usar `n_init > 1` y quedarse con la solución de menor inercia.

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/kmeans_example.png?raw=true' width=500/>
</div>

> **Pregunta**: ¿Qué sucede si se eligen mal los centroides iniciales?


### K-Means++

El K-Means estándar elige los $K$ centroides iniciales **al azar**, lo que puede generar centroides agrupados en la misma región y producir soluciones subóptimas. K-Means++ los elige de forma **deliberadamente dispersa**.

#### Idea central

En lugar de elegir todos los centroides al azar, K-Means++ los va eligiendo **uno a uno**, dando más probabilidad a los puntos que están **lejos** de los centroides ya elegidos:

$$P(x) = \frac{d(x,\, \mathcal{C})^2}{\displaystyle\sum_{x' \in X} d(x',\, \mathcal{C})^2}$$

donde:
- $d(x, \mathcal{C})$ es la distancia del punto $x$ al **centroide más cercano** de los ya elegidos $\mathcal{C}$.
- El denominador normaliza para que todas las probabilidades sumen 1.

**¿Por qué al cuadrado?** Elevar al cuadrado amplifica las diferencias: un punto dos veces más lejos tiene *cuatro veces* más probabilidad de ser elegido. Esto hace que la selección favorezca fuertemente puntos en zonas aún no cubiertas.

#### Algoritmo

1. Elegir el **primer centroide** uniformemente al azar entre todos los puntos.
2. Calcular $d(x, \mathcal{C})^2$ para cada punto restante.
3. Elegir el **siguiente centroide** con probabilidad $P(x)$ — puntos lejanos tienen más chances.
4. Repetir 2–3 hasta tener $K$ centroides bien distribuidos.
5. Ejecutar K-Means estándar desde esa inicialización.

> En scikit-learn, `init='k-means++'` es el **valor por defecto**. El parámetro `n_init` controla cuántas veces se reinicia el proceso completo, quedándose con la mejor solución.

In [ ]:
# Random State permite controlar la aleatoridad.
# Es decir, permite generar los mismos números aleatorios en distintas ejecuciones.
RANDOM_STATE = 99

In [ ]:
from sklearn.cluster import KMeans

# El número de clusters es parámetro. En este caso, es 2.
kmeans = KMeans(
    n_clusters=2,
    random_state=RANDOM_STATE,
    n_init=10,
).fit(scaled_features)

labels = kmeans.labels_

Podemos acceder a los centroides calculados.

In [ ]:
# Clusters calculados por cada observación de entrenamiento
labels

In [ ]:
# Centroides calculados.
kmeans.cluster_centers_

> `inertia_` es exactamente la SSE de la fórmula: $\sum_k \sum_{x \in C_k} d(c_k, x)^2$


In [ ]:
# Un valor menor indica que los puntos están más cerca de sus centroides.
kmeans.inertia_

In [ ]:
# Cuantas features utilizó
kmeans.n_features_in_

In [ ]:
fig_df["kmeans_labels_2"] = pd.Series(kmeans.labels_, dtype=str)
fig = get_scatter(fig_df, "kmeans_labels_2")

# Transformamos los centros que vimos anteriormente a la proyección 2d.
projected_centers = proyector.transform(kmeans.cluster_centers_)

fig.add_trace(
    go.Scatter(
        x=projected_centers[:, 0],
        y=projected_centers[:, 1],
        mode="markers",
        # name="Centros",
        marker_size=12,
        marker_color="LightSlateGray",
        showlegend=False,
    )
)

### Limitaciones de K-Means

- **Asume clusters esféricos**: funciona bien cuando los grupos son convexos y de tamaño similar. Falla con formas irregulares o clusters alargados.
- **Sensible a outliers**: un punto muy alejado desplaza el centroide hacia él, distorsionando el cluster.
- **Requiere misma escala en las features**: si una variable tiene rango $[0, 1]$ y otra $[0, 10^6]$, la segunda domina la distancia euclideana. De ahí la importancia del escalado previo (MinMaxScaler, StandardScaler).
- **Mínimos locales**: distintas inicializaciones pueden dar resultados distintos. Se recomienda usar `n_init > 1` y K-Means++.

> ¿Cambia mucho el resultado si usamos más clusters? Probemos con $K=5$.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init="auto").fit(
    scaled_features
)
labels = kmeans.labels_


fig_df["kmeans_labels_5"] = pd.Series(kmeans.labels_, dtype=str)
fig = get_scatter(fig_df, "kmeans_labels_5")

In [ ]:
# Transformamos los centros que vimos anteriormente a la proyección 2d.
centros_proyectados = proyector.transform(kmeans.cluster_centers_)

fig = get_scatter(fig_df, "kmeans_labels_5")
fig.add_trace(
    go.Scatter(
        x=centros_proyectados[:, 0],
        y=centros_proyectados[:, 1],
        mode="markers",
        name="Centros",
        marker_size=12,
        marker_color="red",
        showlegend=False,
    )
)

> **Pregunta:** ¿Cómo identificamos la cantidad de cluster óptimos?


### Método del Codo

<div align='center'>

![Método del Codo](https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/elbow.png?raw=true)
</div>

La idea es simple: a medida que aumentamos $K$, la inercia siempre disminuye (con más grupos, los puntos quedan más cerca de su centroide). Sin embargo, a partir de cierto $K$ la ganancia marginal se vuelve pequeña. El "codo" de la curva - donde la pendiente cambia bruscamente - indica el $K$ óptimo.

> **Limitación**: el codo no siempre es claro y puede ser subjetivo. En esos casos conviene complementar con el coeficiente de Silhouette.

In [ ]:
intertias = [
    [
        i,
        KMeans(n_clusters=i, random_state=0, n_init="auto")
        .fit(scaled_features)
        .inertia_,
    ]
    for i in range(2, 20)
]

intertias = pd.DataFrame(intertias, columns=["n° clusters", "inertia"])
intertias.head(10)

In [ ]:
px.line(
    intertias,
    x="n° clusters",
    y="inertia",
    title="Método del Codo con K-Means",
    height=600,
)

**Alternativa: Coeficiente de Silhouette**

El coeficiente de silueta mide, para cada punto, **qué tan bien está ubicado dentro de su cluster** comparado con los clusters vecinos. Combina dos ideas:

- **Cohesión** ($a$): ¿qué tan compacto es mi cluster? → promedio de distancias al resto de puntos del mismo cluster. Queremos $a$ pequeño.
- **Separación** ($b$): ¿qué tan lejos estoy del cluster más cercano que no es el mío? → menor distancia promedio a cualquier otro cluster. Queremos $b$ grande.

$$s = \frac{b - a}{\max(a,\, b)}$$

El denominador normaliza el resultado al rango $[-1, 1]$:

| Valor de $s$ | Interpretación |
|---|---|
| $s \approx 1$ | $b \gg a$: el punto está muy cerca de su cluster y lejos de los demás → bien asignado |
| $s \approx 0$ | $a \approx b$: el punto está en la frontera entre dos clusters |
| $s \approx -1$ | $a \gg b$: el punto está más cerca de otro cluster que del propio → mal asignado |

**Ejemplo numérico**: si $a = 0.2$ y $b = 0.8$, entonces $s = \frac{0.8 - 0.2}{0.8} = 0.75$ — el punto está bien clusterizado.

El **coeficiente de silueta global** se obtiene promediando los valores individuales. Cuanto más cercano a 1, mejor es el clustering en general.

> ⚠️ **Limitación**: calcular $s$ para todos los puntos requiere computar todas las distancias entre pares, lo que tiene complejidad $O(n^2)$. En datasets grandes conviene usar `silhouette_score(..., sample_size=N)` para aproximarlo sobre una muestra.


In [ ]:
from sklearn.metrics import silhouette_score

scores = [
    [
        i,
        silhouette_score(
            scaled_features,
            KMeans(n_clusters=i, random_state=RANDOM_STATE, n_init="auto")
            .fit(scaled_features)
            .labels_,
        ),
    ]
    for i in range(2, 20)
]

scores = pd.DataFrame(scores, columns=["n° clusters", "silhouette_score"])
scores.head(10)

In [ ]:
px.line(scores, x="n° clusters", y="silhouette_score")

In [ ]:
from sklearn.metrics import silhouette_samples
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Comparamos dos valores de K para ver cómo se distribuyen los coeficientes por cluster
ks_to_plot = [3, 5]

fig = make_subplots(
    rows=1,
    cols=len(ks_to_plot),
    subplot_titles=[f"K = {k}" for k in ks_to_plot],
    shared_yaxes=False,
)

for col, k in enumerate(ks_to_plot, start=1):
    labels_k = KMeans(
        n_clusters=k, random_state=RANDOM_STATE, n_init="auto"
    ).fit_predict(scaled_features)
    sil_vals = silhouette_samples(scaled_features, labels_k)
    avg = sil_vals.mean()

    y_pos = 0
    for cluster_id in range(k):
        cluster_sil = np.sort(sil_vals[labels_k == cluster_id])
        n = len(cluster_sil)

        fig.add_trace(
            go.Bar(
                x=cluster_sil,
                y=list(range(y_pos, y_pos + n)),
                orientation="h",
                name=f"Cluster {cluster_id}",
                showlegend=(col == 1),
                marker_line_width=0,
            ),
            row=1,
            col=col,
        )
        y_pos += n + 10  # separación entre clusters

    # línea vertical con el coeficiente promedio
    fig.add_vline(
        x=avg,
        line_dash="dash",
        line_color="red",
        row=1,
        col=col,
        annotation_text=f"avg={avg:.2f}",
        annotation_position="top right",
    )

fig.update_layout(
    title="Diagrama de Silhouette por cluster",
    height=500,
    bargap=0,
    xaxis_title="Coeficiente de Silhouette",
    xaxis2_title="Coeficiente de Silhouette",
)
fig.update_yaxes(showticklabels=False)
fig.show()


**¿Cómo leer este gráfico?**

Cada fila es un punto del dataset. Las barras están agrupadas por cluster (un bloque de color por cluster) y ordenadas de menor a mayor coeficiente dentro de cada grupo. La línea roja discontinua es el coeficiente de silueta promedio global.

Un buen clustering se ve así:

- **Barras largas y uniformes**: los puntos del cluster están bien separados del resto → coeficientes cercanos a 1.
- **Todos los bloques superan la línea roja**: los clusters son homogéneamente buenos.
- **Bloques de tamaño similar**: los clusters están balanceados en cantidad de puntos.

Señales de un clustering problemático:

- **Barras cortas o negativas**: puntos mal asignados, probablemente más cercanos a otro cluster.
- **Un bloque que no llega a la línea roja**: ese cluster está "peor" que el promedio — podría indicar que está mezclando puntos de distintos grupos reales.
- **Bloques muy dispares en tamaño**: clustering desequilibrado.


**Más opciones:**

📖 [How to Measure Clustering Performance When There Are No Ground Truth Labels](https://medium.com/@haataa/how-to-measure-clustering-performances-when-there-are-no-ground-truth-db027e9a871c) — Medium

---

## ¿K-Means con datos categóricos?

> **Pregunta ❓**: ¿Podría usar K-Means con datos categóricos, como el género musical, la tonalidad o el modo de una canción?

**No directamente.**

K-Means se basa en calcular **distancias euclideanas** y **medias aritméticas**. Ambas operaciones pierden significado con datos categóricos:

- ¿Cuánto "dista" la categoría `rock` de `jazz`? No hay una respuesta numérica natural.
- ¿Cuál es la "media" entre `rock`, `jazz` y `reggaeton`? No existe.

**Ejemplo concreto**: si codificamos géneros como `rock=1, jazz=2, reggaeton=3`, la distancia entre `rock` y `reggaeton` sería 2, mientras que entre `rock` y `jazz` sería 1 — como si `jazz` fuera "más parecido" al `rock` que el `reggaeton`, lo que es una afirmación arbitraria que depende del orden de codificación.

> ¿Qué alternativas existen?

## K-Modes

[K-Modes](https://pypi.org/project/kmodes) adapta K-Means para trabajar con **datos categóricos** mediante dos cambios fundamentales:

| | **K-Means** | **K-Modes** |
|---|---|---|
| Medida de (di)similitud | Distancia euclideana | Disimilitud de coincidencia simple |
| Representante del cluster | Media aritmética | **Moda** (categoría más frecuente) |
| Función de costo | SSE | Suma de disimilitudes |

### Disimilitud de coincidencia simple (*Simple Matching Dissimilarity*)

Dados dos puntos $x$ e $y$ con $p$ atributos categóricos:

$$d(x, y) = \sum_{j=1}^{p} \mathbf{1}[x_j \neq y_j]$$

Es decir: **contar cuántos atributos son distintos**. Si `x = (rock, mayor, 4/4)` e `y = (jazz, mayor, 4/4)`, entonces $d(x, y) = 1$ (solo difieren en género).

### Algoritmo

El algoritmo es análogo a K-Means, reemplazando media por moda:

1. Inicializar $K$ modas (centroides categóricos).
2. Asignar cada punto al cluster de menor disimilitud.
3. Actualizar cada moda como la **categoría más frecuente** de cada atributo en el cluster.
4. Repetir hasta convergencia.

### En la práctica

Para usar K-Modes con atributos numéricos, es común **discretizarlos** en rangos con significado: por ejemplo, `tempo` → {lento, moderado, rápido}. 

Esto permite que el algoritmo trabaje exclusivamente con categorías y aproveche la disimilitud de coincidencia.

In [ ]:
from kmodes.kmodes import KModes

# --- Discretización de atributos numéricos a categóricos ---
# Convertimos variables continuas con significado musical en rangos interpretables

cat_km = df[["genre", "key", "mode", "time_signature"]].copy().astype(str)

cat_km["tempo_cat"] = pd.cut(
    df["tempo"],
    bins=[0, 90, 130, 999],
    labels=["lento", "moderado", "rápido"],
).astype(str)

cat_km["energy_cat"] = pd.cut(
    df["energy"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["baja", "media", "alta"],
).astype(str)

cat_km["valence_cat"] = pd.cut(
    df["valence"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["negativo", "neutro", "positivo"],
).astype(str)

cat_km["danceability_cat"] = pd.cut(
    df["danceability"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["poco_bailable", "bailable", "muy_bailable"],
).astype(str)

print("Columnas usadas:", cat_km.columns.tolist())
print("Forma del dataset:", cat_km.shape)
cat_km.head()


In [ ]:
km = KModes(n_clusters=3, init="Huang", n_init=10, random_state=RANDOM_STATE)
km.fit(cat_km)

cat_km["cluster"] = km.labels_.astype(str)

# Distribución de géneros dentro de cada cluster
px.histogram(
    cat_km,
    x="genre",
    color="cluster",
    barmode="group",
    title="Distribución de géneros por cluster (K-Modes)",
    labels={"genre": "Género", "cluster": "Cluster"},
)


In [ ]:
px.scatter(
    cat_km.join(pd.DataFrame(projections, columns=["x", "y"])),
    x="x",
    y="y",
    color="cluster",
    hover_data=["genre", "tempo_cat", "energy_cat", "valence_cat", "danceability_cat"],
    title="K-Modes — proyección UMAP (coloreado por cluster)",
)

---

## K-Prototypes

> **Motivación** 💡: En la práctica, la mayoría de los datasets son **mixtos**: tienen columnas numéricas (tempo, loudness, energy...) *y* columnas categóricas (género, tonalidad, modo...). ¿Cómo agrupamos cuando tenemos ambos tipos?

**K-Prototypes** combina K-Means y K-Modes en un único algoritmo para manejar datos mixtos.

### Función de costo

$$C = \sum_{k=1}^{K} \left( \underbrace{\sum_{x \in C_k} \sum_{j \in \text{num}} (x_j - \mu_{kj})^2}_{\text{K-Means: atributos numéricos}} + \gamma \underbrace{\sum_{x \in C_k} \sum_{j \in \text{cat}} \mathbf{1}[x_j \neq \phi_{kj}]}_{\text{K-Modes: atributos categóricos}} \right)$$

donde:
- $\mu_{kj}$: media del atributo numérico $j$ en el cluster $k$ (como K-Means).
- $\phi_{kj}$: moda del atributo categórico $j$ en el cluster $k$ (como K-Modes).
- $\gamma$: parámetro de peso que controla la **importancia relativa** de los atributos categóricos frente a los numéricos.

> **¿Cómo elegir $\gamma$?** Si $\gamma$ es muy pequeño, el algoritmo ignora las columnas categóricas. Si es muy grande, ignora las numéricas. Por defecto, la librería lo estima automáticamente como la desviación estándar media de las columnas numéricas.

### Representante del cluster (*prototipo*)

Cada cluster queda representado por un **prototipo** que combina:
- La **media** de los atributos numéricos.
- La **moda** de los atributos categóricos.

In [ ]:
from kmodes.kprototypes import KPrototypes

# Columnas numéricas y categóricas
num_cols = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "valence",
    "liveness",
    "tempo",
    "loudness",
    "duration_ms",
]
cat_cols = ["key", "mode", "time_signature"]

mixed_data = df[num_cols + cat_cols].copy()
mixed_data[cat_cols] = mixed_data[cat_cols].astype(str)

# Los índices de las columnas categóricas son necesarios para K-Prototypes
cat_indices = [mixed_data.columns.get_loc(c) for c in cat_cols]

kp = KPrototypes(
    n_clusters=3, init="Huang", n_init=3, random_state=RANDOM_STATE, gamma=0.1
)
kp.fit(mixed_data, categorical=cat_indices)

mixed_data["cluster"] = kp.labels_.astype(str)

print("Prototipos (numérico + categórico por cluster):")
for i, proto in enumerate(kp.cluster_centroids_):
    print(f"  Cluster {i}: numérico={proto[0]}, categórico={proto[1]}")

px.scatter(
    mixed_data.join(pd.DataFrame(projections, columns=["x", "y"])),
    x="x",
    y="y",
    color="cluster",
    title="K-Prototypes (datos numéricos + categóricos)",
)

### Resumen: K-Means, K-Modes y K-Prototypes

| | **K-Means** | **K-Modes** | **K-Prototypes** |
|---|---|---|---|
| Tipo de datos | Solo numéricos | Solo categóricos | Mixtos (num + cat) |
| Medida de similitud | Distancia euclideana | Disimilitud de coincidencia | Euclideana + coincidencia ($\gamma$) |
| Representante del cluster | Media | Moda | Media + Moda (prototipo) |
| Librería (Python) | `sklearn` | `kmodes` | `kmodes` |

> **Regla práctica**: Si tienes variables categóricas en tu dataset, codificarlas como números y aplicar K-Means directamente introduce sesgos arbitrarios. Prefiere K-Modes o K-Prototypes según el tipo de datos.

> **Pregunta** 🤔: K-Means asume que los clusters son **esféricos** y de tamaño similar. ¿Qué pasaría si los clusters tuvieran formas irregulares, alargadas o muy distintas densidades? ¿Seguiría funcionando bien?

> **Motivación** 💡: Hasta ahora con K-Means necesitamos saber $K$ de antemano y el algoritmo no tiene concepto de "ruido". ¿Qué pasa cuando los datos tienen **outliers**, formas irregulares, o simplemente no sabemos cuántos clusters hay?

---


## `DBSCAN`
	

Algoritmo de clustering basado en densidad. Ideal para buscar outliers.

- Densidad: Número de puntos en un círculo (hiperesfera de radio $\varepsilon$).
- Idea: Regiones densas representan clusters.

Parámetros: 

- `Eps` ($\varepsilon$): radio de la vecindad de cada punto
- `MinPts`: número mínimo de puntos dentro de esa vecindad para que el punto sea considerado denso.


- Es comúnmente resistente al ruido.
- Problemas en regiones con distintas densidades.

Tipos de puntos:
- **Punto núcleo**: tiene al menos `min_samples` dentro de su `eps`. Es considerado un punto denso.
- **Punto frontera**: tiene menos de `min_samples` dentro de su vecindario, pero está cerca de un núcleo. Está conectado, pero no forma parte del núcleo.
- **Punto ruido**: no es núcleo ni frontera, por lo tanto se considera un punto aislado (outlier).

<div align='center'>
<img src="https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/dbscan.png" width=800 />
</div>

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.3, min_samples=3)
dbscan.fit(scaled_features)

fig_df["dbscan"] = pd.Series(dbscan.labels_, dtype=str)

get_scatter(fig_df, "dbscan")

### Visualizando el parámetro eps

In [ ]:
import numpy as np

epss = np.arange(0.1, 1, 0.2)
epss

In [ ]:
clustering = [
    DBSCAN(eps=eps, min_samples=2).fit_predict(scaled_features) for eps in epss
]

dbscan_labels = pd.DataFrame(np.array(clustering)).T
dbscan_labels.columns = np.round(epss, 3)

dbscan_labels["x"] = projections[:, 0]
dbscan_labels["y"] = projections[:, 1]

dbscan_labels = dbscan_labels.melt(
    id_vars=["x", "y"], var_name="eps", value_name="label"
)
dbscan_labels["label"] = dbscan_labels["label"].astype(str)
dbscan_labels.sample(10)

In [ ]:
fig = px.scatter(
    dbscan_labels,
    x="x",
    y="y",
    facet_row="eps",
    color="label",
    height=1600,
)
fig.show()

In [ ]:
dbscan_labels.loc[dbscan_labels["eps"] == 0.3, "label"].value_counts()

### Eligiendo `eps`: gráfico de k-distancias

Uno de los mayores desafíos de DBSCAN es elegir un buen valor de `eps`. Un valor demasiado pequeño clasifica casi todo como ruido; uno demasiado grande fusiona todo en un único cluster.

El **gráfico de k-distancias** es el método estándar para calibrarlo:

1. Para cada punto, se calcula la distancia a su **vecino más cercano** (excluyendo el propio punto).
2. Se ordenan esas distancias de menor a mayor y se grafican.
3. La curva resultante tiene forma de "codo".

**Ejes del gráfico:**
- **Eje X** (*Puntos ordenados*): índice de orden, no representa ninguna variable del dataset.
- **Eje Y** (*Distancia al 2° vecino*): distancia al vecino más cercano de cada punto. Valores bajos → zona densa; valores altos → punto aislado.

**¿Cómo encontrar el `eps` óptimo?**

Observa la curva de izquierda a derecha:
- Al principio sube despacio y de forma uniforme — esos puntos están en zonas densas.
- En algún punto la curva **cambia de pendiente abruptamente**: en lugar de subir poco a poco, empieza a crecer de forma pronunciada. Ese "salto" visible es el **codo**.
- El valor en el **eje Y** justo en ese codo es el `eps` recomendado: por debajo de él están los puntos densos; por encima, los outliers.

> En el gráfico siguiente el codo aparece alrededor de $\varepsilon \approx 0.35$. Usaremos ese valor.


In [ ]:
from sklearn.neighbors import NearestNeighbors

neighbors = NearestNeighbors(n_neighbors=2).fit(scaled_features)
distances, indices = neighbors.kneighbors(scaled_features)
distances = np.sort(distances[:, 1])

px.line(
    x=range(len(distances)),
    y=distances,
    labels={"x": "Puntos ordenados", "y": "Distancia al 2º vecino"},
    title="Gráfico de k-distancias (para elegir eps)",
).update_traces(line_color="steelblue")


In [ ]:
dbscan_opt = DBSCAN(eps=0.35, min_samples=3).fit(scaled_features)

fig_df["dbscan_opt"] = pd.Series(dbscan_opt.labels_, dtype=str)

n_clusters = len(set(dbscan_opt.labels_)) - (1 if -1 in dbscan_opt.labels_ else 0)
n_noise = (dbscan_opt.labels_ == -1).sum()
print(f"Clusters encontrados: {n_clusters} | Puntos de ruido (-1): {n_noise}")

get_scatter(fig_df, "dbscan_opt").update_layout(
    title=f"DBSCAN con eps=0.35 — {n_clusters} clusters, {n_noise} outliers"
)

---

## Clustering Jerárquico

A diferencia de K-Means, los métodos jerárquicos **no requieren especificar $k$ de antemano** ni asumen ninguna forma particular de los clusters. En cambio, construyen una **jerarquía de agrupaciones** que se puede visualizar como un árbol llamado **dendrograma**.

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/clustering_jerarquico.png' width=400 />
</div>

> **¿Qué es un dendrograma?** Es un diagrama en forma de árbol donde:
> - Las **hojas** (extremos) representan puntos individuales.
> - Las **ramas** representan fusiones de clusters.
> - La **altura** de cada fusión indica qué tan disimilares eran los clusters al unirse: fusiones bajas → muy similares; fusiones altas → muy distintos.
> - Para obtener $k$ clusters, se "corta" el árbol horizontalmente a una altura elegida.

Requieren que exista una **definición de distancia** entre los elementos que se desean agrupar.

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/matriz_distancias.png' width=800 />
</div>


### Tipos

Existen dos estrategias opuestas:

#### Aglomerativo (*bottom-up*)

- Parte con **cada punto como su propio cluster** ($n$ clusters).
- En cada paso, **fusiona los dos clusters más cercanos** según algún criterio de enlace.
- Se detiene cuando queda un solo cluster (o cuando se alcanza el $k$ deseado).
- *Analogía*: empezar con canciones sueltas e ir formando playlists fusionando las más parecidas.

#### Divisivo (*top-down*)

- Parte con **todos los puntos en un único cluster**.
- En cada paso, **divide** un cluster en dos.
- Se detiene cuando cada punto es su propio cluster (o cuando se alcanza $k$).
- *Analogía*: tomar toda la biblioteca musical y dividirla progresivamente en géneros, subgéneros, etc.

> **En la práctica**, casi siempre se usa el enfoque **aglomerativo**: es más eficiente y más fácil de implementar. El resto de la sección se enfoca en él.


### Algoritmo básico Aglomerativo

```
1. Inicializar: cada punto es su propio cluster → n clusters.
2. Calcular la matriz de distancias entre todos los pares de clusters.
3. Repetir:
   a. Encontrar el par de clusters (A, B) con menor distancia.
   b. Fusionar A y B en un nuevo cluster C = A ∪ B.
      → Registrar esta fusión en el dendrograma con su altura = distancia(A, B).
   c. Actualizar la matriz de distancias: reemplazar las filas/columnas
      de A y B por la fila/columna del nuevo cluster C.
4. Hasta que quede un solo cluster.
```

> **Conexión con el dendrograma**: en el paso 3b, cada fusión añade una rama al árbol. La **altura** a la que se une esa rama es la distancia entre los dos clusters en el momento de la fusión. Al final, el dendrograma registra el historial completo de todas las fusiones — elegir $k$ equivale a decidir dónde "cortar" ese árbol.

> **Complejidad**: $O(n^2)$ en memoria (la matriz de distancias) y al menos $O(n^2 \log n)$ en tiempo. Para datasets grandes (> ~10 000 puntos) puede volverse lento o inviable.


### Tipos de Enlace entre Clusters

El **enlace** (*linkage*) define cómo medir la distancia entre dos clusters (no entre dos puntos). Es el ingrediente más importante del algoritmo: cambiarlo puede producir resultados completamente distintos.

| Enlace | Fórmula | Forma típica de clusters | Sensibilidad a outliers | Cuándo usarlo |
|--------|---------|--------------------------|------------------------|---------------|
| **Completo** (`complete`) | $\max\{d(x,y): x \in A, y \in B\}$ | Esféricos, compactos | Baja | Datos bien separados, se quiere compacidad |
| **Simple** (`single`) | $\min\{d(x,y): x \in A, y \in B\}$ | Alargados, cadenas | Muy alta | Clusters con formas irregulares o no convexas |
| **Promedio** (`average`) | $\frac{1}{\lvert A\rvert \lvert B\rvert}\sum_{x \in A}\sum_{y \in B}d(x,y)$ | Intermedios | Media | Balance general, buen punto de partida |
| **Ward** (`ward`) | $\Delta\text{SSE}(A,B) = \text{SSE}(A \cup B) - \text{SSE}(A) - \text{SSE}(B)$ | Esféricos, tamaños similares | Baja | **Recomendado por defecto**: equivale a minimizar varianza intra-cluster |

> La opción elegida puede provocar variaciones gigantescas entre los clusters producidos. Ver los ejemplos a continuación.


In [ ]:
import numpy as np
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import average, complete, single, ward
from sklearn.cluster import AgglomerativeClustering

#### Máx - Enlace Completo (`complete`)
    
<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/agglomerative_max.png' width=400 />
</div>
    
$$\max\{d(x,y): x \in A, y \in B\}$$
    
    
- Solo se agrupan dos clusters si todos sus puntos están suficientemente cerca entre sí.
- Produce clusters esféricos, bien separados, y menos sensibles a outliers.


In [ ]:
sample = scaled_features.sample(20, random_state=RANDOM_STATE)
sample

In [ ]:
# plot the top three levels of the dendrogram
ff.create_dendrogram(
    sample,
    labels=(
        df.loc[sample.index, "artist"] + " - " + df.loc[sample.index, "name"]
    ).values,
    linkagefun=complete,
    orientation="left",
    color_threshold=1.5,
).update_layout(width=1000, height=800)

In [ ]:
model = AgglomerativeClustering(n_clusters=5, linkage="complete")
labels = model.fit_predict(scaled_features)

fig_df["ag_complete"] = pd.Series(model.labels_, dtype=str)

get_scatter(fig_df, "ag_complete")

In [ ]:
pd.Series(model.labels_, dtype=str).value_counts()

#### Mín - Enlace Simple (`single`)

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/agglomerative_min.png' width=400 />
</div>

$$\min\{d(x,y): x \in A, y \in B\}$$

- Agrupa puntos uno por uno a lo largo del camino más corto.
- Resulta en clusters muy alargados.
- Es muy sensible al ruido o a "puentes" de puntos intermedios.


In [ ]:
# plot the top three levels of the dendrogram
ff.create_dendrogram(
    sample,
    labels=(
        df.loc[sample.index, "artist"] + " - " + df.loc[sample.index, "name"]
    ).values,
    linkagefun=single,
    orientation="left",
    color_threshold=0.4,
).update_layout(width=1000, height=800)

In [ ]:
model = AgglomerativeClustering(n_clusters=5, linkage="single")
labels = model.fit_predict(scaled_features)


fig_df["ag_single"] = pd.Series(model.labels_, dtype=str)
get_scatter(fig_df, "ag_single")

In [ ]:
pd.Series(model.labels_, dtype=str).value_counts()

#### Promedio (`average`)

<div align='center'>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/agglomerative_mean.png' width=400 />
</div>

$$\frac{1}{|A||B|}\sum_{x \in A}\sum_{y \in B}d(x,y)$$

- Usa la **distancia promedio** entre todos los pares de puntos de dos clusters.
- No es tan estricto como `complete` ni tan flexible como `single`.
- Produce resultados más balanceados, pero puede ser más lento de calcular.


In [ ]:
# plot the top three levels of the dendrogram
ff.create_dendrogram(
    sample,
    labels=(
        df.loc[sample.index, "artist"] + " - " + df.loc[sample.index, "name"]
    ).values,
    linkagefun=average,
    orientation="left",
    color_threshold=0.8,
).update_layout(width=1000, height=800)

In [ ]:
model = AgglomerativeClustering(n_clusters=5, linkage="average")
labels = model.fit_predict(scaled_features)

fig_df["ag_average"] = pd.Series(model.labels_, dtype=str)
get_scatter(fig_df, "ag_average")

In [ ]:
pd.Series(model.labels_, dtype=str).value_counts()

#### Ward (`ward`)

$$\Delta\text{SSE}(A, B) = \text{SSE}(A \cup B) - \text{SSE}(A) - \text{SSE}(B)$$

Fusiona el par de clusters cuya unión **incrementa menos el SSE total**. Al igual que K-Means, Ward minimiza la varianza intra-cluster 

> Ward sobre datos normalizados suele producir clusters muy similares a K-Means con las mismas inicializaciones.

- Tiende a producir clusters de tamaño similar y forma esférica.
- Es el enlace **más robusto en la práctica**: se recomienda como punto de partida.


In [ ]:
# plot the top three levels of the dendrogram
ff.create_dendrogram(
    sample,
    labels=(
        df.loc[sample.index, "artist"] + " - " + df.loc[sample.index, "name"]
    ).values,
    linkagefun=ward,
    orientation="left",
    color_threshold=0.8,
).update_layout(width=1000, height=800)

In [ ]:
model = AgglomerativeClustering(n_clusters=5, linkage="ward")
labels = model.fit_predict(scaled_features)

fig_df["ag_ward"] = pd.Series(model.labels_, dtype=str)
get_scatter(fig_df, "ag_ward")

In [ ]:
pd.Series(model.labels_, dtype=str).value_counts()

### Resumen: Clustering Jerárquico

**Ventajas:**
- No requiere especificar $k$ a priori: se elige cortando el dendrograma.
- Determinístico: dado el mismo dataset y enlace, siempre produce el mismo resultado.

**Desventajas:**
- **Costoso**: $O(n^2)$ en memoria y $O(n^2 \log n)$ en tiempo. Para $n > 10\,000$ puntos puede ser inviable.
- La mayoría de los esquemas no tienen una función objetivo explícita a minimizar (excepto Ward).
- Diferentes esquemas de enlace presentan problemas y sesgos distintos — la elección del enlace puede cambiar radicalmente los clusters obtenidos.
- Una vez fusionados dos clusters, **no se puede deshacer** la decisión.

> **Recomendación práctica**: usa `ward` como enlace por defecto. Si los clusters son muy alargados o no convexos, prueba `single`. Si quieres clusters compactos y bien separados, `complete` o `average` son buenas alternativas.


> **Reflexión** 🤔: ¿Cuándo preferiríamos clustering jerárquico sobre K-Means? Piensen en escenarios donde la estructura jerárquica de los datos sea informativa por sí misma (ej: taxonomías biológicas, organización de documentos), o cuando queramos explorar múltiples niveles de granularidad sin tener que reejecutar el algoritmo.

El dendrograma revela la **estructura a múltiples escalas** — útil cuando la jerarquía en sí misma es informativa (taxonomías, organización de documentos, filogenia).

> **Motivación** 💡: Tanto K-Means como los métodos jerárquicos hacen una **asignación dura**: cada punto pertenece exactamente a un cluster. ¿Pero qué pasa cuando un punto está en la frontera entre dos grupos? ¿No sería más natural decir "este punto tiene 70% de probabilidad de pertenecer al grupo A y 30% al grupo B"?

## Gaussian Mixture

<div align='center'>
<img src="https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/gaussian-mixture-models-1.png" width=320 />
<p><em>Cada gaussiana cubre una región del espacio — las elipses muestran la forma y orientación definida por su covarianza Σ.</em></p>
</div>
<div align='center'>
<img src="https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/kmeans_vs_gmm_2.png" width=320 />
<p><em>K-Means asigna cada punto al centroide más cercano (límites rígidos); GMM asigna probabilidades — los colores intermedios indican puntos "fronterizos".</em></p>
</div>


Un **Modelo de Mezclas Gaussianas** (*Gaussian Mixture Model*, GMM) es un modelo probabilístico que asume que los datos fueron generados por una **mezcla de $K$ distribuciones gaussianas**, cada una representando un cluster.

La diferencia clave respecto a K-Means es la **asignación blanda** (*soft assignment*):
- K-Means dice "este punto *pertenece* al cluster 3".
- GMM dice "este punto tiene 80% de probabilidad de pertenecer al cluster 3, 15% al cluster 1, y 5% al cluster 2".

Esto lo hace más flexible y más informativo, especialmente cuando los puntos están en zonas de frontera entre clusters.

Formalmente, GMM modela la densidad conjunta de los datos como una suma ponderada de $K$ gaussianas:

$$p(x) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(x \mid \mu_k, \Sigma_k)$$

donde la suma de los pesos cumple $\sum_k \pi_k = 1$ con $\pi_k \geq 0$.


Cada componente gaussiana tiene tres parámetros:

| Parámetro | Símbolo | Significado |
|-----------|---------|-------------|
| **Media** | $\mu_k$ | Centro del cluster $k$ — análogo al centroide de K-Means. |
| **Covarianza** | $\Sigma_k$ | Forma, tamaño y orientación del cluster. Una $\Sigma_k$ diagonal produce elipses alineadas con los ejes; una $\Sigma_k$ completa produce elipses rotadas. |
| **Peso** | $\pi_k$ | Fracción del dataset que pertenece al cluster $k$. Indica qué tan "grande" es ese componente. |


### Algoritmo: Expectation-Maximization (EM)

Para encontrar los parámetros $\{\pi_k, \mu_k, \Sigma_k\}$ que mejor explican los datos, GMM usa el algoritmo **EM**, que alterna entre dos pasos hasta converger:

| Paso | ¿Qué hace? | Analogía con K-Means |
|------|-----------|----------------------|
| **E** (*Expectation*) | Para cada punto, calcula la **probabilidad de pertenecer** a cada cluster usando los parámetros actuales. | Asignación de puntos al centroide más cercano — pero en versión blanda (probabilidades en vez de etiquetas duras). |
| **M** (*Maximization*) | **Reestima** $\mu_k$, $\Sigma_k$ y $\pi_k$ para maximizar la verosimilitud de los datos, usando las probabilidades calculadas en E. | Recalcular los centroides — pero ponderando cada punto por su probabilidad de pertenencia. |

El algoritmo converge cuando los parámetros dejan de cambiar (o la mejora en log-verosimilitud es menor que un umbral). Al igual que K-Means, **puede quedar atrapado en óptimos locales**, por lo que conviene ejecutarlo varias veces con distintas inicializaciones (`n_init` en sklearn).


### Funcionamiento resumido

<div align='center'>
<img src="https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2026-01/11_Clustering/gmm_gif.gif" width=300 />
</div>

1. **Inicializar Parámetros**: Inicializa las medias $\mu_k$, las matrices de covarianza $\Sigma_k$ y los coeficientes de mezcla $\pi_k$ con valores aleatorios o predefinidos.
2. **Calcular probabilidad de pertenencia a cada cluster**: Calcula la probabilidad de que una observación provenga de un cluster $k$.
3. **Reestimar Parámetros**: Actualiza todos los parámetros utilizando las asignaciones encontradas en el paso 2.
4. **Calcular la Log-verosimilitud**: Calcula la log-verosimilitud de los datos dados el modelo.
5. **Verificar Convergencia**: Define un criterio de convergencia. Si el valor de la log-verosimilitud se estabiliza (o si todos los parámetros convergen), detén el proceso. De lo contrario, regresa al Paso 2.

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=5, random_state=32)
gmm.fit(scaled_features)

In [ ]:
# Para obtener las labels
gmm.predict(scaled_features)

In [ ]:
# Para obtener las probabilidaes
gmm.predict_proba(scaled_features)

In [ ]:
# caso particular
gmm.predict_proba(scaled_features)[0]

In [ ]:
# Obtener los centroides de las distribuciones
gmm.means_

In [ ]:
# número de cluster
gmm.n_components

In [ ]:
# pesos
print(gmm.weights_)

In [ ]:
gmm.weights_.sum()

In [ ]:
fig_df["gmm_labels"] = gmm.predict(scaled_features)
fig_gmm = get_scatter(fig_df, "gmm_labels")

# Transformamos los centro que vimos anteriormentes a la proyección 2d.
projected_centers_gmm = proyector.transform(gmm.means_)

fig_gmm.add_trace(
    go.Scatter(
        x=projected_centers_gmm[:, 0],
        y=projected_centers_gmm[:, 1],
        mode="markers",
        marker_size=12,
        marker_color="LightSlateGray",
    )
)

### Selección del número de componentes: BIC y AIC

A diferencia de K-Means, GMM es un modelo probabilístico, lo que permite usar **criterios de información** para elegir $K$ de forma más rigurosa que el método del codo.

| Criterio | Fórmula (simplificada) | Penalización | Tendencia |
|----------|----------------------|-------------|-----------|
| **AIC** | $-2 \ln \hat{L} + 2p$ | Suave | Elige $K$ más grande |
| **BIC** | $-2 \ln \hat{L} + p \ln n$ | Más fuerte (crece con $n$) | Elige $K$ más pequeño |

donde $\hat{L}$ es la log-verosimilitud del modelo ajustado, $p$ el número de parámetros y $n$ el número de puntos.

En ambos casos **menor es mejor**. El $K$ óptimo es aquel en que la curva se aplana — similar al codo del método del codo. Cuando AIC y BIC discrepan, BIC suele ser más conservador y robusto ante sobreajuste.


In [ ]:
bic_aic_scores = [
    [
        k,
        GaussianMixture(n_components=k, random_state=32)
        .fit(scaled_features)
        .bic(scaled_features),
        GaussianMixture(n_components=k, random_state=32)
        .fit(scaled_features)
        .aic(scaled_features),
    ]
    for k in range(2, 15)
]

bic_aic_df = pd.DataFrame(bic_aic_scores, columns=["n° componentes", "BIC", "AIC"])

px.line(
    bic_aic_df.melt(id_vars="n° componentes", var_name="criterio", value_name="score"),
    x="n° componentes",
    y="score",
    color="criterio",
    title="Selección de K en GMM: BIC y AIC",
    height=500,
)

### GMM vs K-Means

| | **K-Means** | **GMM** |
|-|-------------|---------|
| **Tipo de asignación** | Dura (cada punto → un cluster) | Blanda (probabilidades de pertenencia a cada cluster) |
| **Forma de clusters** | Solo esférica (distancia euclídea) | Elipsoidal arbitraria (controlada por $\Sigma_k$) |
| **Tamaño de clusters** | Tendencia a clusters de tamaño similar | Puede manejar clusters de muy distintos tamaños (pesos $\pi_k$) |
| **Selección de $K$** | Método del codo, Silhouette | BIC / AIC |
| **Costo computacional** | Bajo — $O(nKd)$ por iteración | Mayor — estima matrices de covarianza $\Sigma_k$ de tamaño $d \times d$ |
| **Sensibilidad a inicialización** | Sí (usar K-Means++) | Sí (EM puede caer en óptimos locales; usar `n_init > 1`) |
| **Escalabilidad** | Muy buena | Puede ser lento con muchos componentes o dimensiones altas |
| **Interpretabilidad** | Sencilla | Las probabilidades añaden riqueza interpretativa |

> **Regla práctica**: si los clusters tienen formas no esféricas o quieres probabilidades de pertenencia, usa GMM. Si el dataset es grande y la velocidad importa, K-Means es mejor punto de partida.


---

## ¿Cuándo usar cada algoritmo?

La elección del algoritmo depende de las características del problema. Esta guía puede servir de punto de partida:

| Situación | Algoritmo recomendado |
|---|---|
| Sé cuántos clusters quiero, los datos son numéricos y aproximadamente esféricos | **K-Means** |
| No sé cuántos clusters hay y existen outliers o formas irregulares | **DBSCAN** |
| Quiero explorar múltiples niveles de agrupación o la jerarquía es informativa | **Jerárquico (Aglomerativo)** |
| Los clusters pueden solaparse y quiero probabilidades de pertenencia | **GMM** |
| Los datos son categóricos | **K-Modes** |
| Los datos mezclan tipos numéricos y categóricos | **K-Prototypes** |

> **Nota**: en la práctica, siempre conviene probar más de un algoritmo y comparar resultados con métricas como Silhouette o BIC/AIC.

---

## Interpretación de Clusters

Obtener etiquetas de cluster es solo el primer paso. El verdadero desafío es entender **qué significa cada cluster** — darle un nombre, una historia, una identidad.

> Los clusters no tienen nombre por sí solos. Son solo números (0, 1, 2...). El trabajo del analista es **conectar esos números con conocimiento del dominio**.

### ¿Cómo se interpretan los clusters?

Existen varias estrategias complementarias:

| Estrategia | Pregunta que responde | Herramienta típica |
|---|---|---|
| **Perfil de features** | ¿En qué variables se distingue cada cluster? | Heatmap de medias por cluster |
| **Variables externas** | ¿Coincide el cluster con alguna etiqueta conocida? | Distribución de clases, boxplots por cluster |
| **Ejemplos representativos** | ¿Qué casos "típicos" hay en cada cluster? | Punto más cercano al centroide |
| **Nombrado** | ¿Qué nombre le damos a cada grupo? | Interpretación humana |

En esta sección aplicamos las dos primeras a nuestro dataset musical con **K-Means K=3**:

1. **Perfil medio por cluster** (heatmap de features escaladas)
2. **Composición por género** (¿qué géneros dominan cada grupo?)


In [ ]:
# Entrenamos K-Means con K=3 para la interpretación
kmeans_3 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init="auto").fit(
    scaled_features
)
fig_df["kmeans_labels_3"] = pd.Series(kmeans_3.labels_, dtype=str)

get_scatter(fig_df, "kmeans_labels_3")


### 1. Perfil medio por cluster (heatmap)

Calculamos el **promedio de cada feature escalada** dentro de cada cluster y lo visualizamos como un heatmap. Cada fila es un cluster; cada columna, una feature. Los colores muestran si el cluster tiene valores **altos** (cálido) o **bajos** (frío) respecto al promedio global.


In [ ]:
num_features = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "valence",
    "liveness",
    "loudness",
    "tempo",
]

# Perfil medio de features escaladas por cluster de K-Means (K=3)
cluster_profile = (
    scaled_features[num_features]
    .assign(cluster=fig_df["kmeans_labels_3"])
    .groupby("cluster")
    .mean()
    .round(3)
)

px.imshow(
    cluster_profile,
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text_auto=True,
    aspect="auto",
    title="Perfil medio de cada cluster (features escaladas, K-Means K=3)",
    labels={"x": "Feature", "y": "Cluster", "color": "Media escalada"},
    height=320,
)


### 2. Composición por género

¿Los clusters capturan géneros musicales? Visualizamos la **distribución de géneros dentro de cada cluster** como un gráfico de barras apiladas normalizado.

> Si los clusters son buenos, esperamos que cada uno esté dominado por uno o pocos géneros. Si los géneros se distribuyen uniformemente en todos los clusters, las features acústicas no separan bien por género.


In [ ]:
genre_cluster = (
    df[["genre"]]
    .assign(cluster=fig_df["kmeans_labels_3"])
    .groupby(["cluster", "genre"])
    .size()
    .rename("count")
    .reset_index()
)

# Normalizar por cluster → proporciones
genre_cluster["pct"] = genre_cluster.groupby("cluster")["count"].transform(
    lambda x: x / x.sum() * 100
)

px.bar(
    genre_cluster,
    x="cluster",
    y="pct",
    color="genre",
    barmode="stack",
    title="Distribución de géneros por cluster (K-Means K=3)",
    labels={"pct": "% canciones", "cluster": "Cluster", "genre": "Género"},
    height=500,
)


> **Reflexión** 🎓: Con el heatmap y el gráfico de géneros, intenta **ponerle un nombre** a cada uno de los 3 clusters. Por ejemplo: ¿hay un cluster de música enérgica y bailable? ¿Otro de música acústica tranquila? ¿Uno más instrumental? Eso es exactamente lo que se hace en proyectos reales: los clusters no tienen nombre hasta que un humano los interpreta usando evidencia cuantitativa (features) y cualitativa (géneros, ejemplos concretos).


---

## Resumen General: Comparación de Algoritmos de Clustering

| | **K-Means** | **DBSCAN** | **Jerárquico** | **GMM** |
|---|---|---|---|---|
| **Tipo** | Particional | Basado en densidad | Jerárquico | Difuso (probabilístico) |
| **¿Requiere K?** | ✅ Sí | ❌ No | ❌ No (se corta el dendrograma) | ✅ Sí |
| **Forma de clusters** | Esférica | Arbitraria | Depende del enlace | Elipsoidal |
| **Maneja outliers** | ❌ Sensible | ✅ Los etiqueta como ruido | ❌ Sensible | ⚠️ Parcialmente |
| **Tipo de asignación** | Dura | Dura | Dura | Suave (probabilidades) |
| **Escalabilidad** | ✅ Alta | ⚠️ Media | ❌ Baja ($O(n^2)$) | ⚠️ Media |
| **Función objetivo** | SSE (inercia) | — | SSE (solo Ward) | Log-verosimilitud |
| **Hiperparámetros clave** | `n_clusters` | `eps`, `min_samples` | `n_clusters`, `linkage` | `n_components` |



> **Reflexión final** 🎓: No existe un único algoritmo de clustering "mejor". La clave está en entender los supuestos de cada método y validar los resultados con métricas adecuadas y, cuando sea posible, con conocimiento del dominio.